# 02 – Provera chunkova

Automatske provere da li je chunking ispravan, za obe strategije (fixed i section).

In [2]:
import sys, json, re
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir())
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
from config import DATA_PROCESSED, EVAL_QUESTIONS
from chunking import load_jsonl

chunks = {name: load_jsonl(DATA_PROCESSED / f"chunks_{name}.jsonl") for name in ["fixed", "section"]}
questions = json.loads(EVAL_QUESTIONS.read_text(encoding="utf-8"))


def check(name, ok, detail=""):
    """Ispisuje ✅ ili ❌; kod neuspeha prikazuje i detalje (šta tačno nije prošlo)."""
    print(f"{'✅' if ok else '❌'} {name}" + (f"  ->  {detail}" if detail and not ok else ""))
    return ok

In [3]:
rows = []
for name, ch in chunks.items():
    words = pd.Series([len(c["text"].split()) for c in ch])
    rows.append({"strategija": name, "chunk-ova": len(ch), "min": words.min(),
                 "medijana": int(words.median()), "max": words.max(), "ukupno reči": words.sum()})
pd.DataFrame(rows).set_index("strategija")

,chunk-ova,min,medijana,max,ukupno reči
strategija,,,,,
fixed,30,191,300,300,8891
section,99,11,62,274,7441


In [4]:
pd.DataFrame([
    {"strategija": n, **pd.Series([c["type"] for c in ch]).value_counts().to_dict()}
    for n, ch in chunks.items()
]).set_index("strategija").fillna(0).astype(int)

,text,table,faq
strategija,,,
fixed,27,3,0
section,41,28,30


In [5]:
needed_sections = {s for q in questions for s in q["sources"] if not s.startswith("Figure")}
needed_figures = {s for q in questions for s in q["sources"] if s.startswith("Figure")}
OFFICES = {"Prague", "Lisbon", "Krakow", "Austin", "Toronto", "Singapore"}
TABLE_SEP = re.compile(r"^\|[\s\-:|]+\|$")      # red ispod zaglavlja tabele: |-----|-----|

for name, ch in chunks.items():
    print(f"\n--- {name} ---")

    # 1. Pokrivenost: svaka sekcija koju evaluacija traži postoji u bar jednom chunk-u
    covered = {s for c in ch for s in c["sections"]}
    missing = needed_sections - covered
    check("sve sekcije iz evaluacionog skupa postoje", not missing, sorted(missing))

    # 2. Svaka slika iz evaluacije je bar pomenuta ([Figure N: ...])
    referenced = {f"Figure {n}" for c in ch for n in re.findall(r"\[Figure (\d+):", c["text"])}
    missing_f = needed_figures - referenced
    check("sve slike iz evaluacionog skupa su pomenute", not missing_f, sorted(missing_f))

    # 3. Sadržaj (TOC) je uklonjen; njegovi redovi sadrže tab znak
    toc = [c["chunk_id"] for c in ch if "\t" in c["text"] or "## Contents" in c["text"]]
    check("nema linija iz sadržaja (TOC)", not toc, toc)

    # 4. Arhiva (pogl. 17) je označena kao zastarela
    wrong = [c["chunk_id"] for c in ch if c["section"].split(".")[0] == "17" and c["status"] != "superseded"]
    check("poglavlje 17 je označeno kao superseded", not wrong, wrong)

    # 5. Tekst iz arhive ne sme da se nađe u chunk-u koji nije označen kao superseded
    leaked = [(c["chunk_id"], c["status"]) for c in ch
              if any(s.split(".")[0] == "17" for s in c["sections"]) and c["status"] != "superseded"]
    check("arhiva ne 'curi' u chunk-ove sa drugim statusom", not leaked, leaked)

    # 6. Sve kancelarije imaju metapodatak office
    offices = {c["office"] for c in ch if c["office"]}
    check("svih 6 kancelarija ima 'office'", offices == OFFICES, sorted(OFFICES - offices))

    # 7. Tabele nisu presečene: svaka tabela počinje zaglavljem i redom |---|---|
    broken = []
    for c in ch:
        for block in c["text"].split("\n\n"):
            lines = block.strip().splitlines()
            if lines and lines[0].startswith("|") and (len(lines) < 2 or not TABLE_SEP.match(lines[1])):
                broken.append(c["chunk_id"])
    check("nijedna tabela nije presečena", not broken, broken)


--- fixed ---
✅ sve sekcije iz evaluacionog skupa postoje
✅ sve slike iz evaluacionog skupa su pomenute
✅ nema linija iz sadržaja (TOC)
✅ poglavlje 17 je označeno kao superseded
❌ arhiva ne 'curi' u chunk-ove sa drugim statusom  ->  [('fixed-028', 'valid'), ('fixed-029', 'valid')]
❌ svih 6 kancelarija ima 'office'  ->  ['Krakow', 'Singapore']
❌ nijedna tabela nije presečena  ->  ['fixed-011', 'fixed-017', 'fixed-022']

--- section ---
✅ sve sekcije iz evaluacionog skupa postoje
✅ sve slike iz evaluacionog skupa su pomenute
✅ nema linija iz sadržaja (TOC)
✅ poglavlje 17 je označeno kao superseded
✅ arhiva ne 'curi' u chunk-ove sa drugim statusom
✅ svih 6 kancelarija ima 'office'
✅ nijedna tabela nije presečena
